In [ ]:
import requests, json, time
wf = {}
wf['1'] = {'class_type':'UnetLoaderGGUF','inputs':{'unet_name':'ltx-2.3-22b-distilled-Q4_K_S.gguf'}}
wf['2'] = {'class_type':'EmptyLTXVLatentVideo','inputs':{'width':256,'height':256,'length':9,'batch_size':1}}
wf['10'] = {'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'ltxv'}}
wf['3'] = {'class_type':'CLIPTextEncode','inputs':{'text':'red ball','clip':['10',0]}}
wf['4'] = {'class_type':'CLIPTextEncode','inputs':{'text':'','clip':['10',0]}}
wf['11'] = {'class_type':'LTXVConditioning','inputs':{'positive':['3',0],'negative':['4',0],'frame_rate':24}}
wf['12'] = {'class_type':'LTXVScheduler','inputs':{'steps':4,'max_shift':2.05,'base_shift':0.95,'stretch':True,'terminal':0.1}}
wf['13'] = {'class_type':'KSamplerSelect','inputs':{'sampler_name':'euler'}}
wf['14'] = {'class_type':'RandomNoise','inputs':{'noise_seed':42}}
wf['15'] = {'class_type':'BasicGuider','inputs':{'model':['1',0],'conditioning':['11',0]}}
wf['5'] = {'class_type':'SamplerCustomAdvanced','inputs':{'guider':['15',0],'sampler':['13',0],'sigmas':['12',0],'noise':['14',0],'latent_image':['2',0]}}
wf['9'] = {'class_type':'VAELoader','inputs':{'vae_name':'LTX23_video_vae_bf16.safetensors'}}
wf['6'] = {'class_type':'VAEDecode','inputs':{'samples':['5',0],'vae':['9',0]}}
wf['8'] = {'class_type':'SaveImage','inputs':{'images':['6',0],'filename_prefix':'ltx_test'}}
r = requests.post('http://127.0.0.1:8188/prompt', json={'prompt':wf})
print('Submit:', r.status_code, r.text[:500])


In [ ]:
# Wait and check output
import os, glob
if r.status_code == 200:
    pid = r.json().get('prompt_id','')
    print('Prompt ID:', pid)
    print('Waiting for generation...')
    for i in range(60):
        time.sleep(5)
        h = requests.get(f'http://127.0.0.1:8188/history/{pid}').json()
        if pid in h:
            print('Done! Outputs:', json.dumps(h[pid].get('outputs',{}), indent=2)[:500])
            break
        print(f'  {(i+1)*5}s...')
    # Check files
    for d in ['/content/ComfyUI/output', '/content/ComfyUI/temp']:
        files = glob.glob(os.path.join(d, '**', '*'), recursive=True)
        new = [f for f in files if os.path.getmtime(f) > time.time() - 300]
        if new:
            print(f'New files in {d}:')
            for f in new:
                print(f'  {f} ({os.path.getsize(f)} bytes)')
